## Dataset Analysis

In [2]:
import pandas as pd

# LOAD DATA
df = pd.read_csv("/content/jan to may police violation_anonymized791b166.csv")

print("="*50)
print("DATASET SHAPE")
print("="*50)
print(df.shape)

print("\n")

print("="*50)
print("COLUMN NAMES")
print("="*50)
print(df.columns.tolist())

print("\n")

# DATETIME
df["created_datetime"] = pd.to_datetime(
    df["created_datetime"],
    format="mixed",
    utc=True,
    errors="coerce"
)
df["hour"] = df["created_datetime"].dt.hour
df["day_of_week"] = df["created_datetime"].dt.day_name()
df["month"] = df["created_datetime"].dt.month

print("="*50)
print("DATE RANGE")
print("="*50)
print(df["created_datetime"].min())
print(df["created_datetime"].max())

print("\n")

print("="*50)
print("TOP 20 POLICE STATIONS")
print("="*50)
print(df["police_station"].value_counts().head(20))

print("\n")

print("="*50)
print("TOP 20 JUNCTIONS")
print("="*50)
print(df["junction_name"].value_counts().head(20))

print("\n")

print("="*50)
print("VIOLATIONS BY HOUR")
print("="*50)
print(df["hour"].value_counts().sort_index())

print("\n")

print("="*50)
print("VIOLATIONS BY DAY")
print("="*50)
print(df["day_of_week"].value_counts())

print("\n")

print("="*50)
print("CREATING ML DATASET")
print("="*50)

agg = (
    df.groupby(
        [
            "police_station",
            "junction_name",
            "day_of_week",
            "hour"
        ]
    )
    .size()
    .reset_index(name="violations")
)

print("Aggregated Shape:")
print(agg.shape)

print("\nSample Rows:")
print(agg.head(20))

print("\n")

print("="*50)
print("TARGET STATISTICS")
print("="*50)
print(agg["violations"].describe())

DATASET SHAPE
(298450, 24)


COLUMN NAMES
['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']


DATE RANGE
2023-11-09 19:11:46+00:00
2024-04-08 17:30:46+00:00


TOP 20 POLICE STATIONS
police_station
Upparpet                  34468
Shivajinagar              28044
Malleshwaram              22200
HAL Old Airport           20819
City Market               17646
Vijayanagara              14652
Rajajinagar               10998
Kodigehalli               10916
Magadi Road                8558
Jeevanbheemanagar          6736
K.R. Pura                  6546
Halasuru Gate              6294
Mahadevapura               

# Parking Impact Intelligence System

In [ ]:
# Parking Impact Intelligence System
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

X = agg[
    [
        "police_station",
        "junction_name",
        "day_of_week",
        "hour"
    ]
]

y = agg["violations"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

cat_features = [
    0,
    1,
    2
]

model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

preds = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    preds
)

print("MAE =", mae)

0:	learn: 40.2467577	total: 94.2ms	remaining: 1m 34s
200:	learn: 23.1465018	total: 3.34s	remaining: 13.3s
400:	learn: 20.1075433	total: 8.47s	remaining: 12.7s
600:	learn: 18.0871352	total: 11.8s	remaining: 7.81s
800:	learn: 16.5469912	total: 14.1s	remaining: 3.5s
999:	learn: 15.5065520	total: 16.4s	remaining: 0us
MAE = 9.493268794677713


In [6]:
fi = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": model.get_feature_importance()
    }
)

print(
    fi.sort_values(
        "importance",
        ascending=False
    )
)

          feature  importance
0  police_station   41.397684
3            hour   28.558695
1   junction_name   23.951343
2     day_of_week    6.092279


## Hotspot Detection

In [7]:
import pandas as pd
import numpy as np

# =========================
# CLEAN VIOLATION TYPES
# =========================

df["violation_clean"] = (
    df["violation_type"]
    .astype(str)
    .str.upper()
)

# =========================
# SEVERITY SCORING
# =========================

def get_severity(v):

    v = str(v)

    if "DOUBLE PARKING" in v:
        return 10

    elif "PARKING IN A MAIN ROAD" in v:
        return 9

    elif "PARKING ON FOOTPATH" in v:
        return 8

    elif "PARKING NEAR" in v:
        return 7

    elif "NO PARKING" in v:
        return 6

    elif "WRONG PARKING" in v:
        return 5

    else:
        return 3

df["severity_score"] = (
    df["violation_clean"]
    .apply(get_severity)
)

# =========================
# HOTSPOT ANALYSIS
# =========================

hotspots = (
    df.groupby(
        [
            "police_station",
            "junction_name"
        ]
    )
    .agg(
        violations=("id","count"),
        avg_severity=("severity_score","mean")
    )
    .reset_index()
)

# =========================
# IMPACT SCORE
# =========================

hotspots["impact_score"] = (
    hotspots["violations"]
    *
    hotspots["avg_severity"]
)

# =========================
# REMOVE NO JUNCTION
# =========================

hotspots = hotspots[
    hotspots["junction_name"] != "No Junction"
]

# =========================
# SORT
# =========================

hotspots = hotspots.sort_values(
    "impact_score",
    ascending=False
)

print("="*60)
print("TOP 20 CONGESTION HOTSPOTS")
print("="*60)

print(
    hotspots[
        [
            "police_station",
            "junction_name",
            "violations",
            "impact_score"
        ]
    ].head(20)
)

# =========================
# RISK LEVEL
# =========================

def risk_level(score):

    if score >= 15000:
        return "CRITICAL"

    elif score >= 7000:
        return "HIGH"

    elif score >= 3000:
        return "MEDIUM"

    else:
        return "LOW"

hotspots["risk_level"] = (
    hotspots["impact_score"]
    .apply(risk_level)
)

# =========================
# ENFORCEMENT RECOMMENDATION
# =========================

def recommendation(risk):

    if risk == "CRITICAL":
        return "Deploy 3 officers and continuous monitoring"

    elif risk == "HIGH":
        return "Deploy 2 officers during peak hours"

    elif risk == "MEDIUM":
        return "Periodic enforcement patrol"

    else:
        return "Routine monitoring"

hotspots["recommendation"] = (
    hotspots["risk_level"]
    .apply(recommendation)
)

print("\n")
print("="*60)
print("TOP 20 ENFORCEMENT PRIORITIES")
print("="*60)

print(
    hotspots[
        [
            "police_station",
            "junction_name",
            "impact_score",
            "risk_level",
            "recommendation"
        ]
    ].head(20)
)

# =========================
# SAVE
# =========================

hotspots.to_csv(
    "hotspot_recommendations.csv",
    index=False
)

print("\nSaved hotspot_recommendations.csv")

TOP 20 CONGESTION HOTSPOTS
    police_station                               junction_name  violations  \
283   Shivajinagar              BTP051 - Safina Plaza Junction       15298   
291       Upparpet                     BTP040 - Elite Junction       10718   
293       Upparpet             BTP044 - Sagar Theatre Junction       10049   
71     City Market                 BTP082 - KR Market Junction       10109   
286   Shivajinagar            BTP211 - Central Street Junction        5373   
299       Upparpet                  BTP058 - Subbanna Junction        4830   
320   Vijayanagara            BTP020 - Hosahalli Metro Station        4035   
298       Upparpet                 BTP057 - Anand Rao Junction        3641   
121  Halasuru Gate          BTP080 - NR Road, SP Road Junction        3283   
294       Upparpet           BTP045 - Danvanthri Road Junction        3181   
226   Malleshwaram      BTP001 - 10th Cross, Dr. Rajkumar Road        2812   
266    Rajajinagar               BTP0

In [8]:
import pandas as pd

# =========================
# CREATE TOMORROW DATASET
# =========================

future_rows = []

top_hotspots = hotspots.head(20)

hours = [17,18,19,20]

for _, row in top_hotspots.iterrows():

    for h in hours:

        future_rows.append({

            "police_station":
            row["police_station"],

            "junction_name":
            row["junction_name"],

            "day_of_week":
            "Sunday",

            "hour":
            h
        })

future_df = pd.DataFrame(
    future_rows
)

# =========================
# PREDICT
# =========================

future_df["expected_violations"] = (
    model.predict(future_df)
)

future_df = future_df.sort_values(
    "expected_violations",
    ascending=False
)

print(
    future_df.head(30)
)

future_df.to_csv(
    "tomorrow_hotspots.csv",
    index=False
)

print(
    "\nSaved tomorrow_hotspots.csv"
)

   police_station                       junction_name day_of_week  hour  \
14    City Market         BTP082 - KR Market Junction      Sunday    19   
22       Upparpet          BTP058 - Subbanna Junction      Sunday    19   
15    City Market         BTP082 - KR Market Junction      Sunday    20   
13    City Market         BTP082 - KR Market Junction      Sunday    18   
23       Upparpet          BTP058 - Subbanna Junction      Sunday    20   
21       Upparpet          BTP058 - Subbanna Junction      Sunday    18   
0    Shivajinagar      BTP051 - Safina Plaza Junction      Sunday    17   
1    Shivajinagar      BTP051 - Safina Plaza Junction      Sunday    18   
2    Shivajinagar      BTP051 - Safina Plaza Junction      Sunday    19   
12    City Market         BTP082 - KR Market Junction      Sunday    17   
3    Shivajinagar      BTP051 - Safina Plaza Junction      Sunday    20   
10       Upparpet     BTP044 - Sagar Theatre Junction      Sunday    19   
11       Upparpet     BTP

In [9]:
import folium
from folium.plugins import HeatMap

m = folium.Map(
    location=[
        df["latitude"].mean(),
        df["longitude"].mean()
    ],
    zoom_start=11
)

heat_data = df[
    [
        "latitude",
        "longitude"
    ]
].values.tolist()

HeatMap(
    heat_data,
    radius=12
).add_to(m)

m.save(
    "parking_heatmap.html"
)

print("Saved parking_heatmap.html")

Saved parking_heatmap.html


In [10]:
print(
    df["vehicle_number"]
    .nunique()
)

print(
    df["vehicle_number"]
    .value_counts()
    .head(20)
)

231890
vehicle_number
FKN00GL4424     55
FKN00GL3514     42
FKN00GL9771     41
FKN00GL17863    41
FKN00GL2906     35
FKN00GL15265    34
FKN00GL14092    34
FKN00GL19337    30
FKN00GL1875     30
FKN00GL9852     29
FKN00GL15753    28
FKN00GL1118     28
FKN00GL1175     28
FKN00GL8564     27
FKN00GL8041     27
FKN00GL20818    26
FKN00GL0420     26
FKN00GL8361     26
FKN00GL24601    26
FKN00GL1498     25
Name: count, dtype: int64


## Repeat Offender Analysis

In [11]:
repeat_offenders = (
    df.groupby("vehicle_number")
    .size()
    .reset_index(name="violations")
)

repeat_offenders = (
    repeat_offenders
    .sort_values(
        "violations",
        ascending=False
    )
)

print(
    repeat_offenders.head(25)
)

repeat_offenders.to_csv(
    "repeat_offenders.csv",
    index=False
)

print(
    "\nSaved repeat_offenders.csv"
)

       vehicle_number  violations
170590    FKN00GL4424          55
160585    FKN00GL3514          42
229374    FKN00GL9771          41
87623    FKN00GL17863          41
153901    FKN00GL2906          35
59027    FKN00GL15265          34
46128    FKN00GL14092          34
103860   FKN00GL19337          30
97390     FKN00GL1875          30
230264    FKN00GL9852          29
20331     FKN00GL1175          28
14061     FKN00GL1118          28
64405    FKN00GL15753          28
210369    FKN00GL8041          27
216122    FKN00GL8564          27
213889    FKN00GL8361          26
149001   FKN00GL24601          26
417       FKN00GL0420          26
120178   FKN00GL20818          26
157559   FKN00GL32387          25
209511    FKN00GL7963          25
55899     FKN00GL1498          25
149968   FKN00GL25480          25
148834    FKN00GL2445          24
152944    FKN00GL2819          24

Saved repeat_offenders.csv


## Enforcement Priority Index (EPI)

In [13]:
# =========================
# PEAK HOUR FACTOR
# =========================

hour_counts = (
    df.groupby("hour")
    .size()
)

peak_hour_count = (
    hour_counts.max()
)

df["peak_factor"] = (
    df["hour"]
    .map(hour_counts)
    /
    peak_hour_count
)

# =========================
# REPEAT OFFENDER FACTOR
# =========================

vehicle_counts = (
    df.groupby("vehicle_number")
    .size()
)

df["repeat_factor"] = (
    df["vehicle_number"]
    .map(vehicle_counts)
)

# normalize

df["repeat_factor"] = (
    df["repeat_factor"]
    /
    df["repeat_factor"].max()
)

# =========================
# EPI
# =========================

df["EPI"] = (
    df["severity_score"]
    *
    (1 + df["peak_factor"])
    *
    (1 + df["repeat_factor"])
)

epi_hotspots = (
    df.groupby(
        [
            "police_station",
            "junction_name"
        ]
    )["EPI"]
    .sum()
    .reset_index()
)

epi_hotspots = (
    epi_hotspots[
        epi_hotspots["junction_name"]
        != "No Junction"
    ]
)

epi_hotspots = (
    epi_hotspots.sort_values(
        "EPI",
        ascending=False
    )
)

print(
    epi_hotspots.head(20)
)

    police_station                               junction_name            EPI
283   Shivajinagar              BTP051 - Safina Plaza Junction  156969.324044
291       Upparpet                     BTP040 - Elite Junction  106117.272303
293       Upparpet             BTP044 - Sagar Theatre Junction  101395.385432
71     City Market                 BTP082 - KR Market Junction   85429.702611
286   Shivajinagar            BTP211 - Central Street Junction   51517.143115
299       Upparpet                  BTP058 - Subbanna Junction   45022.478040
320   Vijayanagara            BTP020 - Hosahalli Metro Station   40443.117074
298       Upparpet                 BTP057 - Anand Rao Junction   33135.953536
121  Halasuru Gate          BTP080 - NR Road, SP Road Junction   32382.009969
294       Upparpet           BTP045 - Danvanthri Road Junction   29735.194088
266    Rajajinagar               BTP027 - Modi Bridge Junction   26276.369492
226   Malleshwaram      BTP001 - 10th Cross, Dr. Rajkumar Road  

In [14]:
top_junctions = [
    "BTP051 - Safina Plaza Junction",
    "BTP040 - Elite Junction",
    "BTP044 - Sagar Theatre Junction",
    "BTP082 - KR Market Junction"
]

coords = (
    df[
        df["junction_name"].isin(top_junctions)
    ]
    .groupby("junction_name")
    [
        ["latitude","longitude"]
    ]
    .mean()
)

print(coords)

                                  latitude  longitude
junction_name                                        
BTP040 - Elite Junction          12.976597  77.576602
BTP044 - Sagar Theatre Junction  12.974939  77.578896
BTP051 - Safina Plaza Junction   12.981212  77.608695
BTP082 - KR Market Junction      12.964424  77.577241


## Risk Classification

In [15]:
import folium



m = folium.Map(

    location=[12.9716, 77.5946],

    zoom_start=12

)



top20 = epi_hotspots.head(20)



for _, row in top20.iterrows():



    coords = (

        df[

            df["junction_name"] ==

            row["junction_name"]

        ][["latitude","longitude"]]

        .mean()

    )



    folium.CircleMarker(

        location=[

            coords["latitude"],

            coords["longitude"]

        ],

        radius=8,

        popup=f"""

        {row['junction_name']}

        <br>

        EPI: {round(row['EPI'])}

        """,

        color="red",

        fill=True

    ).add_to(m)



m.save("risk_map.html")



print("Saved risk_map.html")



Saved risk_map.html


In [16]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# =========================
# CREATE RISK LABELS
# =========================

agg_cls = agg.copy()

agg_cls["risk"] = np.where(
    agg_cls["violations"] >= 50,
    "HIGH",
    np.where(
        agg_cls["violations"] >= 15,
        "MEDIUM",
        "LOW"
    )
)

print(
    agg_cls["risk"].value_counts()
)

# =========================
# FEATURES
# =========================

X = agg_cls[
    [
        "police_station",
        "junction_name",
        "day_of_week",
        "hour"
    ]
]

y = agg_cls["risk"]

# =========================
# SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# CAT FEATURES
# =========================

cat_features = [
    0,
    1,
    2
]

# =========================
# MODEL
# =========================

model_cls = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=200
)

model_cls.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

# =========================
# PREDICT
# =========================

preds = model_cls.predict(X_test)

# =========================
# METRICS
# =========================

acc = accuracy_score(
    y_test,
    preds
)

print("\nAccuracy:")
print(acc)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        preds
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        preds
    )
)

risk
LOW       12257
MEDIUM     3225
HIGH       1408
Name: count, dtype: int64
0:	learn: 1.0640181	total: 19.4ms	remaining: 19.3s
200:	learn: 0.4224474	total: 17.2s	remaining: 1m 8s
400:	learn: 0.3688383	total: 30.5s	remaining: 45.6s
600:	learn: 0.3313274	total: 43.6s	remaining: 28.9s
800:	learn: 0.2986562	total: 56.9s	remaining: 14.1s
999:	learn: 0.2716071	total: 1m 9s	remaining: 0us

Accuracy:
0.8179396092362344

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.74      0.61      0.67       282
         LOW       0.87      0.95      0.91      2451
      MEDIUM       0.56      0.42      0.48       645

    accuracy                           0.82      3378
   macro avg       0.72      0.66      0.68      3378
weighted avg       0.80      0.82      0.81      3378


Confusion Matrix:
[[ 171   19   92]
 [   5 2321  125]
 [  54  320  271]]


In [17]:
# =========================
# SAVE ALL IMPORTANT FILES
# =========================

# 1. EPI Hotspots
epi_hotspots.to_csv(
    "epi_hotspots.csv",
    index=False
)

# 2. Repeat Offenders
repeat_offenders.to_csv(
    "repeat_offenders.csv",
    index=False
)

# 3. Tomorrow Predictions
future_df.to_csv(
    "tomorrow_hotspots.csv",
    index=False
)

# 4. Hotspot Recommendations
hotspots.to_csv(
    "hotspot_recommendations.csv",
    index=False
)

# 5. Save Regression Model
model.save_model(
    "parking_forecast_model.cbm"
)

# 6. Save Classifier Model
model_cls.save_model(
    "parking_risk_classifier.cbm"
)

print("ALL FILES SAVED")

ALL FILES SAVED
